# Session 3: memory arithmetic and the input pipeline

Exploration only. The service depends on `nw/semantic`; this notebook imports it.

## Memory arithmetic, from the real parameter count

In [ ]:
from nw.semantic.model import ModelSpec, build_encoder, apply_lora, count_parameters
model, tok = build_encoder(ModelSpec())
total, _ = count_parameters(model)
print(f"parameters: {total/1e6:.1f}M")
for name, bytes_per in [("weights fp32", 4), ("+ gradients", 8), ("+ Adam moments", 16)]:
    print(f"{name:16s} {total * bytes_per / 1e9:.2f} GB")
model = apply_lora(model, ModelSpec())
total, trainable = count_parameters(model)
print(f"LoRA trainable: {trainable/1e6:.2f}M ({100*trainable/total:.2f}%), optimizer state {trainable*8/1e6:.0f} MB")

## Where the time goes: padding

In [ ]:
import time, torch
from nw.semantic.data import load_rows, make_loader
rows = load_rows("../data/tickets.jsonl", "train")[:512]
for fixed in (False, True):
    from nw.semantic.data import Collate
    loader = make_loader(rows, tok, batch_size=32)
    if fixed:
        loader.collate_fn = lambda b, _c=Collate(tok): {**_c(b), "input_ids": torch.nn.functional.pad(_c(b)["input_ids"], (0, 256 - _c(b)["input_ids"].shape[1]))}
    t0 = time.perf_counter(); tokens = 0
    for batch in loader:
        tokens += batch["input_ids"].numel()
    print(f"{'fixed 256' if fixed else 'dynamic'} padding: {tokens:,} tokens through the encoder, {time.perf_counter()-t0:.2f}s to tokenise")

## Per-head metrics on the test split, from the exported graph

In [ ]:
import json, numpy as np
from nw.semantic.export import OnnxEncoder, load_finetuned
from nw.semantic.data import TAGS, multi_hot
from nw.triage.features import PRIORITIES, ticket_text
from sklearn.metrics import confusion_matrix, f1_score
rows = load_rows("../data/tickets.jsonl", "test")
model, tok, meta = load_finetuned("../artifacts/semantic")
onnx = OnnxEncoder("../artifacts/semantic/model.int8.onnx", tok, meta["max_length"])
thr = np.load("../artifacts/semantic/tag_thresholds.npy")
tag_p, prio_l = [], []
for i in range(0, len(rows), 64):
    a, b, _ = onnx.run([ticket_text(r["subject"], r["body"]) for r in rows[i:i+64]])
    tag_p.append(1/(1+np.exp(-a))); prio_l.append(b)
tag_p, prio_l = np.concatenate(tag_p), np.concatenate(prio_l)
y_prio = np.array([PRIORITIES.index(r["priority"]) for r in rows]); y_tags = np.stack([multi_hot(r["tags"]).numpy() for r in rows])
print("priority confusion (rows true):"); print(confusion_matrix(y_prio, prio_l.argmax(1)))
per_tag = f1_score(y_tags, tag_p >= thr, average=None, zero_division=0)
for t, f in sorted(zip(TAGS, per_tag), key=lambda x: x[1])[:8]:
    print(f"weakest tag {t:14s} F1 {f:.2f}  n={int(y_tags[:, TAGS.index(t)].sum())}")

## Questions to bring back

- Which tags are weakest, and are they rare or ambiguous?
- Where do German tickets land in the priority confusion?